In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
%cd drive/MyDrive

/content/drive/MyDrive


In [ ]:
import torch
import math
from seq2seq_models import preprocess_text, tokenize_data, pad_or_truncate, Vocab, encode, Embedding
from sequential_models import Linear, Tanh, BasicRNN, LSTMCell, GRUCell

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Text preprocessing

In [ ]:
dataset_path = "fra.txt"
src, tgt = [], []

with open(dataset_path) as file_object:
  for i, line in enumerate(file_object):
    result = preprocess_text(line)
    tokenize_data(result, src, tgt)

src = [pad_or_truncate(s) for s in src]
tgt = [pad_or_truncate(t) for t in tgt]
tgt = [["<bos>"] + t for t in tgt]
eng_vocab = Vocab(src)
fr_vocab = Vocab(tgt)

lookup_fxn = lambda sentence, vocab: [encode(vocab, s) for s in sentence]
src_lookup = [lookup_fxn(s, eng_vocab) for s in src]
tgt_lookup = [lookup_fxn(t, fr_vocab) for t in tgt]

src_lookup = torch.tensor(src_lookup, dtype=torch.int32)
tgt_lookup = torch.tensor(tgt_lookup, dtype=torch.int32)

target_label = tgt_lookup[:, 1:]
decoder_input = tgt_lookup[:, :-1]

label_valid_len = (target_label != fr_vocab["<pad>"]).type(torch.int32).sum(1)
decoder_valid_len = (decoder_input != fr_vocab["<pad>"]).type(torch.int32).sum(1)
src_valid_len = (src_lookup != eng_vocab["<pad>"]).type(torch.int32).sum(1)

In [ ]:
class RNNEncoder():
  def __init__(self,
               vocab_size, #number of unique tokens
               feature_size, #number of embedding features
               n_neurons, #number of neurons in the RNN layers
               ):
    self.vocab_size = vocab_size
    self.feature_size = feature_size
    self.neurons = n_neurons

    self.embedding = Embedding(vocab_size, feature_size)

    self.rnn1 = BasicRNN(feature_size, n_neurons)


    #look at hidden state initialization
    self.rnn2 = BasicRNN(n_neurons, n_neurons)


  def __call__(self, x):
    h1 = torch.zeros((x.shape[0], self.neurons))

    dense_input = self.embedding(x)
    # [batch_size, seq_len, feature_size]
    output1, h1 = self.rnn1(dense_input, h1)
    # [seq_len, batch_size, n_hidden], [batch_size, n_hidden]
    output1 = output1.permute(1, 0, 2)
    h2 = torch.zeros((output1.shape[0], self.neurons))
    # [batch_size, seq_len, n_hidden], [batch_size, n_hidden]
    output2, h2 = self.rnn2(output1, h2)
    #[num_of_layers, batch_size, n_hidden], [seq_len, batch_size, n_hidden]
    return (h1, h2), output2

  def __repr__(self):
    rep = f"Encoder(\nEmbedding={self.embedding.embedding.shape},\nRNN=({self.feature_size, self.neurons}), \nRNN=({self.neurons, self.neurons})\n)"
    return rep

  def parameters(self):
    params = self.embedding.parameters() + self.rnn1.parameters() + self.rnn2.parameters()
    return params

In [ ]:
class AdditiveAttention():
  def __init__(self, key_hidden_state, query_hidden_state, new_hidden_state):
    #shape of keys and values = [seq_len, batch_size, n_hidden]
    #hidden state of decoder(query) = [batch_size, n_hidden]
    self.Wk = Linear(key_hidden_state, new_hidden_state)
    self.Wq = Linear(query_hidden_state, new_hidden_state)
    self.Wv = Linear(new_hidden_state, 1)
    self.tanh = Tanh()

  def __repr__(self):
    return f"AdditiveAttention()"

  def __call__(self, keys, queries, values, valid_lens):
    features = self.Wk(keys) + self.Wq(queries)
    #features = [seq_len, batch_size, new_hidden_state]
    scores = self.Wv(self.tanh(features))
    #scores = [seq_len, batch_size, 1]
    scores = scores.permute(1, 2, 0)
    #scores = [batch_size, 1, seq_len]
    values = values.permute(1, 0, 2)
    #values = [batch_size, seq_len, hidden_state]
    self.weights = masked_softmax(scores.squeeze(1), valid_lens)
    self.weights = self.weights.unsqueeze(1)
    #self.weights = [batch_size, 1, seq_len]

    return torch.bmm(self.weights, values)
    #return [batch_size, 1, new_hidden_state]

  def parameters(self):
    return self.Wk.parameters() + self.Wq.parameters() + self.Wv.parameters()

In [ ]:
class RNNAttentionDecoder():
  def __init__(self, feature_size, vocab_size, n_neurons, new_hidden_state, encoder_neurons):
    self.n_neurons = n_neurons
    self.embeddings = Embedding(vocab_size, feature_size)
    self.feature_size = feature_size
    self.vocab_size = vocab_size
    self.rnn1 = BasicRNN(feature_size + encoder_neurons, n_neurons)
    self.rnn2 = BasicRNN(n_neurons, n_neurons)
    self.attention = None
    self.linear = Linear(n_neurons, vocab_size)
    self.new_hidden = new_hidden_state

  def __call__(self, x, h_x, encoder_ouput, x_valid_lens):
    h1, h2 = h_x
    if self.attention is None:
      self.attention = AdditiveAttention(
          encoder_ouput.shape[-1],
          h2.shape[-1],
          self.new_hidden,
      )
    outputs = []
    seq = x.shape[1] #seq_len
    for t in range(seq):
      context = self.attention(encoder_ouput, h2, encoder_ouput, x_valid_lens)

      dense_input = self.embeddings(x[:, t]).unsqueeze(1)

      input_x = torch.cat((dense_input, context), -1)


      output1, h1 = self.rnn1(input_x, h1)
      output1 = output1.permute(1, 0, 2)
      output2, h2 = self.rnn2(output1, h2)

      output2 = output2.squeeze(0)
      outputs.append(output2)

    outputs = torch.stack(outputs)
    outputs = outputs.permute(1, 0, 2)
    logits = self.linear(outputs)
    return logits



  def __repr__(self):
    rep = f"Bahdanau Decoder(\nEmbedding={self.embeddings.embedding.shape},\nBahdanauAttention(), \nRNN=({self.feature_size, self.n_neurons}), \nRNN=({self.n_neurons, self.n_neurons}), \nLinear({self.n_neurons, self.vocab_size}) \n)"
    return rep

  def parameters(self):
    params = self.embeddings.parameters() + self.rnn1.parameters() + self.rnn2.parameters() + self.attention.parameters() + self.linear.parameters()
    return params


In [ ]:
batch_size = 32

#encoder parameters
encoder_vocab_size = len(eng_vocab)
encoder_feature_size = 20
encoder_n_neurons = 10

#decoder parameters
decoder_vocab_size = len(fr_vocab)
decoder_feature_size = 20
decoder_n_neurons = 10
decoder_new_hidden = 20

encoder = RNNEncoder(
    encoder_vocab_size,
    encoder_feature_size,
    encoder_n_neurons
)

decoder = RNNAttentionDecoder(
    decoder_feature_size,
    decoder_vocab_size,
    decoder_n_neurons,
    decoder_new_hidden,
    encoder_n_neurons,
)

decoder.attention = AdditiveAttention(
         encoder_n_neurons,
          decoder_n_neurons,
          decoder_new_hidden,
      )


In [ ]:
import torch.nn.functional as F
epochs = 100
params = encoder.parameters() + decoder.parameters()

for p in params:
  p.requires_grad = True

for epoch in range(epochs):
  idxs = torch.randint(0, batch_size + 1, (batch_size, ))
  #training data
  src_data = src_lookup[idxs]
  #input to decoder
  d_input = decoder_input[idxs]
  #target labels
  labels = target_label[idxs]

  #valid_lengths
  src_lens = src_valid_len[idxs]

  (h1, h2), encoder_output = encoder(src_data)
  #print(logits)
  logits = decoder(d_input, (h1, h2), encoder_output, src_lens)
  b, s, v = logits.shape
  logits = logits.reshape(b*s, -1)
  #use .long() on target values else cross_entropy will not work
  outs = labels.reshape(b*s).long()
  loss = F.cross_entropy(logits, outs)

  for p in params:
    p.grad = None

  loss.backward()

  lr = 0.101

  for p in params:
    p.data += -lr*p.grad

  print(loss.item())






In [ ]:
def masked_softmax(X, valid_len):
  """
  Mask <pad> tokens
  """
  def _mask(X, valid_len, value=0):
    #X is 3D, valid is 2D or 1D
    #print(X.shape)
    #print(valid_len.shape)
    maxlen = X.shape[1]
    mask = torch.arange(maxlen) < valid_len.unsqueeze(-1)
    #print(mask.shape)
    X[~mask] = value
    return X

  if valid_len is None:
    return torch.nn.functional.softmax(X, dim=-1)
  else:
    shape = X.shape
    if valid_len.dim() == 1:
      valid_len = valid_len.repeat_interleave(shape[1])
    else:
      #flatten
      valid_len = valid_len.reshape(-1)

    X = _mask(X.reshape(-1, X.shape[-1]), valid_len, value=-1e6)
    return torch.nn.functional.softmax(X.reshape(shape), dim=-1)


In [ ]:
class DotProductAttention():
  def __init__(self):
    pass

  def __repr__(self):
    return f"DotProductAttention()"

  def __call__(self, queries, keys, values, valid_lens=None):
    d = queries.shape[-1]
    scores = torch.bmm(queries, keys.transpose(1,2))/math.sqrt(d)
    weights = masked_softmax(scores, valid_lens)
    return torch.bmm(weights, values)

In [ ]:
class MultiHeadAttention():
  def __init__(self, query_size, key_size, value_size, new_hidden, num_heads):
    self.Wq = Linear(query_size, new_hidden)
    self.Wk = Linear(key_size, new_hidden)
    self.Wv = Linear(value_size, new_hidden)
    self.Wo = Linear(new_hidden, new_hidden)
    self.heads = num_heads
    self.attention = DotProductAttention()

  def transpose_qkv(self, X):
    batch_size, seq_len, hidden_size = X.shape
    #[batch_size, seq_len, num_heads, new_hidden/num_heads]
    X = X.reshape(batch_size, seq_len, self.heads, -1)
    #[batch_size, num_heads, seq_len, new_hidden/num_heads]
    X = X.permute(0, 2, 1, 3)
    #[batch_size*num_heads, seq_len, new_hidden/num_heads]
    X = X.reshape(-1, seq_len, X.shape[3])
    return X

  def transpose_output(self, X):
    #[batch_size*num_heads, seq_len, new_hidden/num_heads]
    #[batch_size, num_head, seq_len, new_hidden/num_heads]
    X = X.reshape(-1, self.heads, X.shape[1], X.shape[2])
    #[batch_size, seq_len, num_heads, new_hidden/num_heads]
    X = X.permute(0, 2, 1, 3)
    #[batch_size,seq_len, new_hidden]
    X = X.reshape(X.shape[0], X.shape[1], -1)
    return X


  def __call__(self, keys, values, queries, valid_lens):
    queries = self.transpose_qkv(self.Wq(queries))
    keys = self.transpose_qkv(self.Wk(keys))
    values = self.transpose_qkv(self.Wv(values))

    if valid_lens is not None:
      valid_lens = torch.repeat_interleave(valid_lens, self.heads)

    outputs = self.attention(queries, keys, values, valid_lens)
    outputs = self.transpose_output(outputs)
    return self.Wo(outputs)

  def __repr__(self):
    return "MultiHeadAttention()"

  def parameters(self):
    params = self.Wq.parameters() + self.Wk.parameters() + self.Wv.parameters + self.Wo.parameters()
    return params


In [ ]:
x = torch.randn((32, 10, 15))
lens = torch.randint(0, 11, (32, ))
z = x.shape[-1]
multi_attention = MultiHeadAttention(z, z, z, 30, 5)
outs = multi_attention(x, x, x, lens)
print(outs.shape)

torch.Size([32, 10, 30])


In [ ]:
class PositionalEncoding():
  def __init__(self, num_hiddens, maxlen=1000):
    self.P = torch.zeros((1, maxlen, num_hiddens))
    X = torch.arange(maxlen, dtype=torch.float32).reshape(-1, 1)/torch.pow(10000, torch.arange(0, num_hiddens, 2, dtype=torch.float32)/num_hiddens)
    self.P[:,:, 0::2] = torch.sin(X)
    self.P[:, :, 1::2] = torch.cos(X)


  def __call__(self, X):
    return X + self.P[:, X.shape[1], :]
    pass

  def __repr__(self):
    return "PositionalEncoding()"


Attention is all you need!

In [ ]:
class PositionWiseFFN():
  def __init__(self, input_hidden, ffn_hidden):
    self.l1 = Linear(input_hidden, ffn_hidden)
    self.relu = torch.nn.ReLU()
    self.l2 = Linear(ffn_hidden, input_hidden)


  def __call__(self, X):
    return self.l2(self.relu(self.l1(X)))

  def __repr__(self):
    return "PositionWiseFFN()"

  def parameters(self):
    return self.l1.parameters() + self.l2.parameters()

In [ ]:
def batch_norm(X,gamma, beta, moving_mean, moving_variance, eps, momentum):
  if not torch.is_grad_enabled():
    X_hat = (X-moving_mean)/torch.sqrt(moving_variance + eps)
  else:
    mean = X.mean(dim=0)
    var = ((X-mean)**2).mean(dim=0)
    X_hat = (X-mean)/torch.sqrt(var + eps)
    moving_mean = (1-momentum) * moving_mean + momentum * mean
    moving_var = (1-momentum) * moving_mean + momentum * mean
    Y = gamma * X_hat + beta
  return Y, moving_mean, moving_variance

In [ ]:
class BatchNorm():
  def __init__(self, num_features, num_hiddens):
    shape = (1, num_features)
    self.gamma = torch.ones(shape)
    self.beta = torch.zeros(shape)
    self.moving_mean = torch.zeros(shape)
    self.moving_var = torch.ones(shape)


  def __call__(self, X):
    Y, self.moving_mean, self.moving_var = batch_norm(X, self.gamma, self.beta, self.moving_mean, self.moving_var, 1e-5, 0.1)
    return Y

  def __repr__(self):
    return "BatchNorm()"

  def parameters(self):
    return [self.gamma] + [self.beta]

In [2]:
class Dropout():
  def __init__(self, dropout, training=False):
    self.dropout = dropout
    self.training = training

  def __call__(self, X):
    if not self.training:
      self.dropout = 0
    if self.dropout == 1:
      return torch.zeros_like(X)
    mask = (torch.rand(X.shape) > self.dropout).float()
    return mask * X/(1-self.dropout)

  def __repr__(self):
    pass


SyntaxError: incomplete input (1773919895.py, line 2)

In [ ]:
class LayerNorm():
  def __init__(self):
    pass

  def __call__(self):
    pass

  def __repr__(self):
    pass

